# README
1. there are two kinds of scanning
    - zigzag: quicker scan but may have drift borders
    - linear: slower scan but procude high quality scanning image
    - Therefore, it is recommended to use zigzag scan to locate and navigate to area of interest and then use linear scan to produce useful scanning result.
2. there are two kinds of data plotting
    - zigzag: plot using data from zigzag scan
    - linear: plot using data from linear scan
3. When plotting, there are two options:
    - "plotZigzag2DScan" or "plotLinear2DScan": interpolate data and then plot
    - "plotZigzag2DSCanPcolormesh" or "plotLinear2DScanPcolormesh": plot directly from raw data
4. xyz.goStepsForward and xyz.goStepsReverse are switched for z-axis (#3).

In [2]:
#scanning

%load_ext autoreload
%autoreload 2
from auspex.instruments import SR865
from rensci import RenSciDriver
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd
from datetime import datetime
import os
import array

PIEZO_HOST = "172.31.255.99"
PIEZO_PORT = 6002

lockin = SR865()
lockin.connect('TCPIP0::172.31.255.252')

print(f"Connected to {lockin.name} \n\n X= {lockin.x}, Y = {lockin.y}")

xyz = RenSciDriver(PIEZO_HOST, PIEZO_PORT)
xyz.connect()

def refocus(channel=1, nsteps=200, stepsize=10, lockin_time_constant=0.03):
    x=[]
    r=[]
    xyz.goStepsReverse(channel, int(nsteps/2)*stepsize)
    for i in range(nsteps):
        time.sleep(lockin_time_constant * 5)  # Wait for lock-in to stabilize
        r.append(lockin.r)
        x.append(xyz.getPosition(channel))
        xyz.goStepsForward(channel, stepsize)
    r = np.array(r)
    x = np.array(x)
    xyz.wait_open_loop_done()
    return x[np.argmax(r)], x,r
    
def getStepDistance(channel=2, stepSize=10, nsteps=10):
    step = []
    distance = []

    # Record the starting position
    startPosition = xyz.getPosition(channel)

    if channel == 3:
        for i in range(nsteps):
            xyz.goStepsReverse(channel, stepSize)
            step.append((i + 1) * stepSize)
            distance.append(xyz.getPosition(channel) - startPosition)
    else:
        for i in range(nsteps):
            xyz.goStepsForward(channel, stepSize)
            step.append((i + 1) * stepSize)
            distance.append(xyz.getPosition(channel) - startPosition)

    # Linear fit: distance = slope * step + intercept
    coeffs = np.polyfit(step, distance, 1)
    slope = coeffs[0]  # microns per step unit

    # Distance per single step (stepSize=1)
    distancePerStep = abs(slope)  # mm/step
    xyz.goPosition(channel, startPosition)  # Return to original position
    return distancePerStep

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
auspex-ERROR: 2026-06-06 00:38:42,697 ----> Could not initialize interface for TCPIP0::172.31.255.252.
Connected to SR865 Lockin Amplifier 

 X= 1.0, Y = 1.0


TimeoutError: timed out

In [ ]:
# 06/05/2026 - Lingwen
# zigzag scan
# precise feasure 2D scan test , scan 13

#lock-in setting
lockin.amplitude = 0.15
lockin.frequency = 150*1e3
lockin.time_constant = 3e-5
lockin.sensitivity = 500*1e-6
lockin.phase = 0
lockin.filter_slope = 18

# Parameters (defined first)
targetResolution = 0.5
ystepsize = targetResolution/0.3        # scanning resolution; smaller = higher res, slower, usually 0.3 micron/step
zstepsize = targetResolution/0.1         # scanning resolution; smaller = higher res, slower, usually 0.1 micron/step
scanYSize = 0.032   # total scan size in Y (mm)
scanZSize = 0.018   # total scan size in Z (mm)

# Initialize position
xyz.goPosition(2, -1.083) #set y position at the center of the scan
xyz.goPosition(1, 4.592440)
xyz.goPosition(3, -2.475) #set z position at the center of the scan
startXPosition = xyz.getPosition(1)
startYPosition = xyz.getPosition(2)
startZPosition = xyz.getPosition(3)
print(f"YLocation: {startYPosition}")

# Calibrate step distance and compute number of steps
print("\n>> Calibrating step distance...")
yOneStepDistance = getStepDistance(channel=2, stepSize=ystepsize, nsteps=10)
zOneStepDistance = getStepDistance(channel=3, stepSize=zstepsize, nsteps=10)
zstepsize = yOneStepDistance/zOneStepDistance*ystepsize  # adjust z stepsize to match y stepsize in physical distance
zOneStepDistance = getStepDistance(channel=3, stepSize=zstepsize, nsteps=10)
n_y = int(round(scanYSize / (yOneStepDistance*ystepsize)))
n_z = int(round(scanZSize / (zOneStepDistance*zstepsize)))
yOneStepDistanceUm = yOneStepDistance * 1e-3
zOneStepDistanceUm = zOneStepDistance * 1e-3
print(f"   Y step distance: {yOneStepDistanceUm:.3f} µm/step")
print(f"   Z step distance: {zOneStepDistanceUm:.3f} µm/step")
print(f"  Y Resolution (µm/step): {yOneStepDistance * ystepsize:.3f}")
print(f"  Z Resolution (µm/step): {zOneStepDistance * zstepsize:.3f}")
print(f"   Total steps: {n_y} in Y, {n_z} in Z")

# Refocus at the center of the scan area
print("\n>> Refocusing at scan center...")
print("   Coarse refocus...")
xyz.goPosition(1, refocus(channel=1, nsteps=10, stepsize=8,lockin_time_constant=lockin.time_constant)[0])
print("   Fine refocus...")
xyz.goPosition(1, refocus(channel=1, nsteps=10, stepsize=3,lockin_time_constant=lockin.time_constant)[0])
print("   Final refocus...")
x0 = refocus(channel=1, nsteps=10, stepsize=1,lockin_time_constant=lockin.time_constant)[0]
xyz.goPosition(1, x0)
print(f">> Refocused at x = {x0:.3f} mm")

# xyz.goStepsReverse(2, steps=stepsize*int(n_y/2))
# xyz.goStepsReverse(3, steps=stepsize*int(n_z/2))

r = np.zeros((n_z, n_y))
y = np.zeros((n_z, n_y))
z = np.zeros((n_z, n_y))

total_steps = n_y * n_z
last_reported = -1

# ── Scan time estimate ────────────────────────────────────
from datetime import datetime, timedelta
scan_start_time = datetime.now()
estimated_minutes = total_steps / 1500 * 15
estimated_end_time = scan_start_time + timedelta(minutes=estimated_minutes)
print(f"\n========== SCAN TIME ESTIMATE ==========")
print(f"  Total points       : {total_steps}")
print(f"  Estimated duration : {estimated_minutes:.1f} min  ({estimated_minutes/60:.2f} hrs)")
print(f"  Start time         : {scan_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Estimated end time : {estimated_end_time.strftime('%Y-%m-%d %H:%M:%S')}  (rough)")
print(f"========================================\n")

print(f">> Starting 2D scan ({n_y} x {n_z} = {total_steps} points)...")
xyz.goPosition(2, startYPosition - scanYSize/2)
xyz.goPosition(3, startZPosition - scanZSize/2)

updated_3pct  = False
updated_10pct = False

for i in range(n_z):
    xyz.goStepsReverse(3, steps=zstepsize)
    for j in range(n_y):
        if i % 2 == 0:
            xyz.goStepsForward(2, steps=ystepsize)
        else:
            xyz.goStepsReverse(2, steps=ystepsize)
        time.sleep(lockin.time_constant * 5)
        r[i,j] = lockin.r
        y[i,j] = xyz.getPosition(2)
        z[i,j] = xyz.getPosition(3)

        pct = int((i * n_y + j + 1) / total_steps * 100)
        milestone = (pct // 1) * 1
        if milestone > last_reported:
            print(f"   Progress: {milestone}%  (z step {i+1}/{n_z}, y step {j+1}/{n_y})", end='\r')
            last_reported = milestone

        # Refined estimate after 3%
        if not updated_3pct and pct >= 3:
            elapsed_so_far = (datetime.now() - scan_start_time).total_seconds()
            total_estimated_seconds = elapsed_so_far / pct * 100  # (current% - 0%) / elapsed = rate
            remaining_seconds = total_estimated_seconds - elapsed_so_far
            refined_end = datetime.now() + timedelta(seconds=remaining_seconds)
            print(f"\n   [3% update]  Refined duration : {total_estimated_seconds/60:.1f} min")
            print(f"               Refined end time : {refined_end.strftime('%H:%M:%S')}")

            updated_3pct = True

        # Refined estimate after 10%
        if not updated_10pct and pct >= 10:
            elapsed_so_far = (datetime.now() - scan_start_time).total_seconds()
            total_estimated_seconds = elapsed_so_far / pct * 100
            remaining_seconds = total_estimated_seconds - elapsed_so_far
            refined_end = datetime.now() + timedelta(seconds=remaining_seconds)
            print(f"\n   [10% update] Refined duration : {total_estimated_seconds/60:.1f} min")
            print(f"               Refined end time : {refined_end.strftime('%H:%M:%S')}")
            
            updated_10pct = True

print("\n>> Scan complete.")
scan_end_time = datetime.now()
elapsed = scan_end_time - scan_start_time
print(f"   Start time   : {scan_start_time.strftime('%H:%M:%S')}")
print(f"   End time     : {scan_end_time.strftime('%H:%M:%S')}")
print(f"   Elapsed time : {str(elapsed).split('.')[0]}")

# Save to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(os.getcwd(), "scanData")
os.makedirs(save_dir, exist_ok=True)  # create folder if it doesn't exist

filename = os.path.join(save_dir, f"scan_yz_{timestamp}.csv")

df = pd.DataFrame({
    'y_mm': y.flatten(),
    'z_mm': z.flatten(),
    'r':    r.flatten(),
    'n_y': n_y,
    'n_z': n_z,
    'start_x_mm': startXPosition,
    'start_y_mm': startYPosition,
    'start_z_mm': startZPosition,
})

df.to_csv(filename, index=False)
print(f">> Saved {len(df)} points to {filename}")

In [ ]:
# 06/05/2026 - Lingwen
# linear scan
# precise feasure 2D scan test , scan 14

#lock-in setting
lockin.amplitude = 0.15
lockin.frequency = 150*1e3
lockin.time_constant = 3e-5
lockin.sensitivity = 500*1e-6
lockin.phase = 0
lockin.filter_slope = 18

# Parameters (defined first)
targetResolution = 0.3
ystepsize = targetResolution/0.3        # scanning resolution; smaller = higher res, slower, usually 0.3 micron/step
zstepsize = targetResolution/0.1         # scanning resolution; smaller = higher res, slower, usually 0.1 micron/step
scanYSize = 0.015   # total scan size in Y (mm)
scanZSize = 0.0085  # total scan size in Z (mm)

# Initialize position
xyz.goPosition(2, -1.072) #set y position at the center of the scan
xyz.goPosition(1, 4.592440)
xyz.goPosition(3, -2.476) #set z position at the center of the scan
startXPosition = xyz.getPosition(1)
startYPosition = xyz.getPosition(2)
startZPosition = xyz.getPosition(3)
print(f"YLocation: {startYPosition}")

# Calibrate step distance and compute number of steps
print("\n>> Calibrating step distance...")
yOneStepDistance = getStepDistance(channel=2, stepSize=ystepsize, nsteps=10)
zOneStepDistance = getStepDistance(channel=3, stepSize=zstepsize, nsteps=10)
zstepsize = yOneStepDistance/zOneStepDistance*ystepsize  # adjust z stepsize to match y stepsize in physical distance
zOneStepDistance = getStepDistance(channel=3, stepSize=zstepsize, nsteps=10)
n_y = int(round(scanYSize / (yOneStepDistance*ystepsize)))
n_z = int(round(scanZSize / (zOneStepDistance*zstepsize)))
yOneStepDistanceUm = yOneStepDistance * 1e-3
zOneStepDistanceUm = zOneStepDistance * 1e-3
print(f"   Y step distance: {yOneStepDistanceUm:.3f} µm/step")
print(f"   Z step distance: {zOneStepDistanceUm:.3f} µm/step")
print(f"  Y Resolution (µm/step): {yOneStepDistance * ystepsize:.3f}")
print(f"  Z Resolution (µm/step): {zOneStepDistance * zstepsize:.3f}")
print(f"   Total steps: {n_y} in Y, {n_z} in Z")

# Refocus at the center of the scan area
print("\n>> Refocusing at scan center...")
print("   Coarse refocus...")
xyz.goPosition(1, refocus(channel=1, nsteps=10, stepsize=8,lockin_time_constant=lockin.time_constant)[0])
print("   Fine refocus...")
xyz.goPosition(1, refocus(channel=1, nsteps=10, stepsize=3,lockin_time_constant=lockin.time_constant)[0])
print("   Final refocus...")
x0 = refocus(channel=1, nsteps=10, stepsize=1,lockin_time_constant=lockin.time_constant)[0]
xyz.goPosition(1, x0)
print(f">> Refocused at x = {x0:.3f} mm")

# xyz.goStepsReverse(2, steps=stepsize*int(n_y/2))
# xyz.goStepsReverse(3, steps=stepsize*int(n_z/2))

r = np.zeros((n_z, n_y))
y = np.zeros((n_z, n_y))
z = np.zeros((n_z, n_y))

total_steps = n_y * n_z
last_reported = -1

# ── Scan time estimate ────────────────────────────────────
from datetime import datetime, timedelta
scan_start_time = datetime.now()

# Time per point: lock-in settle + measurement
time_per_point = lockin.time_constant * 3  # seconds
# Extra time per Z row: goPosition for Z + goPosition for Y return
time_per_zrow = 0.5  # seconds, adjust based on how fast goPosition is
estimated_seconds = total_steps * time_per_point + n_z * time_per_zrow
estimated_minutes = estimated_seconds / 60

estimated_end_time = scan_start_time + timedelta(seconds=estimated_seconds)
print(f"\n========== SCAN TIME ESTIMATE ==========")
print(f"  Total points       : {total_steps}")
print(f"  Time per point     : {time_per_point*1e3:.1f} ms")
print(f"  Estimated duration : {estimated_minutes:.1f} min  ({estimated_minutes/60:.2f} hrs)")
print(f"  Start time         : {scan_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Estimated end time : {estimated_end_time.strftime('%Y-%m-%d %H:%M:%S')}  (rough)")
print(f"========================================\n")

print(f">> Starting 2D scan ({n_y} x {n_z} = {total_steps} points)...")
xyz.goPosition(2, startYPosition - scanYSize/2)
xyz.goPosition(3, startZPosition - scanZSize/2)

updated_3pct  = False
updated_10pct = False
nextZPosition = startZPosition - scanZSize/2  # start from scan edge

for i in range(n_z):
    xyz.goStepsReverse(3, steps=zstepsize)
    xyz.goPosition(2, startYPosition - scanYSize/2)
    for j in range(n_y):
        xyz.goStepsForward(2, steps=ystepsize)
        time.sleep(lockin.time_constant * 3)
        r[i,j] = lockin.r
        y[i,j] = xyz.getPosition(2)
        z[i,j] = xyz.getPosition(3)

        pct = int((i * n_y + j + 1) / total_steps * 100)
        milestone = (pct // 1) * 1
        if milestone > last_reported:
            print(f"   Progress: {milestone}%  (z step {i+1}/{n_z}, y step {j+1}/{n_y})", end='\r')
            last_reported = milestone

        # Refined estimate after 3%
        if not updated_3pct and pct >= 3:
            elapsed_so_far = (datetime.now() - scan_start_time).total_seconds()
            total_estimated_seconds = elapsed_so_far / pct * 100  # (current% - 0%) / elapsed = rate
            remaining_seconds = total_estimated_seconds - elapsed_so_far
            refined_end = datetime.now() + timedelta(seconds=remaining_seconds)
            print(f"\n   [3% update]  Refined duration : {total_estimated_seconds/60:.1f} min")
            print(f"               Refined end time : {refined_end.strftime('%H:%M:%S')}")

            updated_3pct = True

        # Refined estimate after 10%
        if not updated_10pct and pct >= 10:
            elapsed_so_far = (datetime.now() - scan_start_time).total_seconds()
            total_estimated_seconds = elapsed_so_far / pct * 100
            remaining_seconds = total_estimated_seconds - elapsed_so_far
            refined_end = datetime.now() + timedelta(seconds=remaining_seconds)
            print(f"\n   [10% update] Refined duration : {total_estimated_seconds/60:.1f} min")
            print(f"               Refined end time : {refined_end.strftime('%H:%M:%S')}")
            
            updated_10pct = True

print("\n>> Scan complete.")
scan_end_time = datetime.now()
elapsed = scan_end_time - scan_start_time
print(f"   Start time   : {scan_start_time.strftime('%H:%M:%S')}")
print(f"   End time     : {scan_end_time.strftime('%H:%M:%S')}")
print(f"   Elapsed time : {str(elapsed).split('.')[0]}")

# Save to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(os.getcwd(), "scanData")
os.makedirs(save_dir, exist_ok=True)  # create folder if it doesn't exist

filename = os.path.join(save_dir, f"scan_yz_{timestamp}.csv")

df = pd.DataFrame({
    'y_mm': y.flatten(),
    'z_mm': z.flatten(),
    'r':    r.flatten(),
    'n_y': n_y,
    'n_z': n_z,
    'start_x_mm': startXPosition,
    'start_y_mm': startYPosition,
    'start_z_mm': startZPosition,
})

df.to_csv(filename, index=False)
print(f">> Saved {len(df)} points to {filename}")


In [ ]:
# Data plotting
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

y,z,r = np.loadtxt("scanData/scan_yz_20260604_122343.csv", delimiter=",", skiprows=1, unpack=True)
mask = (y != 0)

# zigzag scan plotter
def plotZigzag2DScan(filepath, resolution=200, cmap='inferno',targetYPositionOffset = 0, targetZPositionOffset = 0):
    # Load data
    df = pd.read_csv(filepath)
    y = df['y_mm'].values
    z = df['z_mm'].values
    r = df['r'].values
    startY = df['start_y_mm'].values[0]
    startZ = df['start_z_mm'].values[0]

    # Mask zeros and convert to microns
    mask = (y != 0)
    y_um = (y[mask] - y[mask].min()) * 1000
    z_um = (z[mask] - z[mask].min()) * 1000
    r = r[mask]

    # Starting position in micron coordinates
    startY_um = (startY - y[mask].min()) * 1000
    startZ_um = (startZ - z[mask].min()) * 1000

    # Interpolate onto regular grid
    y_grid, z_grid = np.meshgrid(np.linspace(y_um.min(), y_um.max(), resolution),
                                  np.linspace(z_um.min(), z_um.max(), resolution))
    R_grid = griddata((y_um, z_um), r, (y_grid, z_grid), method='linear')

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(R_grid,
               extent=[y_um.min(), y_um.max(), z_um.min(), z_um.max()],
               origin='lower',
               aspect='equal',
               cmap=cmap)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Photovoltage (V)')

    # Mark starting position
    ax.plot(startY_um, startZ_um, 'w+', markersize=15, markeredgewidth=3)
    ax.plot(startY_um, startZ_um, 'r*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Start')
    ax.legend(fontsize=8)

    if targetZPositionOffset!= 0 or targetYPositionOffset != 0:
        # Mark target position
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'w+', markersize=15, markeredgewidth=3)
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'b*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Target')
        ax.legend(fontsize=8)

    ax.set_xlabel('Y Position (µm)')
    ax.set_ylabel('Z Position (µm)')
    ax.set_title(f'2D Scan: Photovoltage Map\n{filepath}')
    plt.tight_layout()
    plt.show()

    print(f"   Total points loaded: {len(r)}")
    print(f"   Y range: {y_um.min():.2f} to {y_um.max():.2f} µm")
    print(f"   Z range: {z_um.min():.2f} to {z_um.max():.2f} µm")
    print(f"   Scan center position: Y={startY:.3f} µm, Z={startZ:.3f} µm")
    if targetZPositionOffset!= 0 or targetYPositionOffset != 0:
        print(f"   Scan target position: Y={startY + targetYPositionOffset*1e-3:.3f} µm, Z={startZ + targetZPositionOffset*1e-3:.3f} µm")

def plotZigzag2DScanPcolormesh(filepath, cmap='viridis',targetYPositionOffset = 0, targetZPositionOffset = 0):
    # Load data
    df = pd.read_csv(filepath)
    y = df['y_mm'].values
    z = df['z_mm'].values
    r = df['r'].values
    n_y = int(df['n_y'].values[0])
    n_z = int(df['n_z'].values[0])
    startY = df['start_y_mm'].values[0]
    startZ = df['start_z_mm'].values[0]

    print(f"   Grid: {n_y} x {n_z} = {n_y*n_z}, data points: {len(r)}")

    Y = y.reshape(n_z, n_y).copy()
    Z = z.reshape(n_z, n_y).copy()
    R = r.reshape(n_z, n_y).copy()

    # Snake correction
    Y[1::2, :] = Y[1::2, ::-1]
    Z[1::2, :] = Z[1::2, ::-1]
    R[1::2, :] = R[1::2, ::-1]

    # Sort every row by actual Y position
    for i in range(n_z):
        sort_idx = np.argsort(Y[i, :])
        Y[i, :] = Y[i, sort_idx]
        Z[i, :] = Z[i, sort_idx]
        R[i, :] = R[i, sort_idx]

    # Convert to microns
    Y_um = (Y - Y.min()) * 1000
    Z_um = (Z - Z.min()) * 1000
    startY_um = (startY - y.min()) * 1000
    startZ_um = (startZ - z.min()) * 1000

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.pcolormesh(Y_um, Z_um, R, shading='auto', cmap=cmap)
    ax.set_aspect('equal')  # ✅ add this
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Photovoltage (V)')

    # Mark starting position
    ax.plot(startY_um, startZ_um, 'w+', markersize=15, markeredgewidth=3)
    ax.plot(startY_um, startZ_um, 'r*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Start')
    ax.legend(fontsize=8)

    if targetZPositionOffset!= 0 or targetYPositionOffset != 0:
        # Mark target position
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'w+', markersize=15, markeredgewidth=3)
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'b*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Target')
        ax.legend(fontsize=8)

    ax.set_xlabel('Y Position (µm)')
    ax.set_ylabel('Z Position (µm)')
    ax.set_title(f'2D Scan: Photovoltage Map\n{filepath}')
    plt.tight_layout()
    plt.show()

    print(f"   Y range: {Y_um.min():.2f} to {Y_um.max():.2f} µm")
    print(f"   Z range: {Z_um.min():.2f} to {Z_um.max():.2f} µm")
    print(f"   Scan center position: Y={startY:.3f} mm, Z={startZ:.3f} mm")
    if targetZPositionOffset!= 0 or targetYPositionOffset != 0:
        print(f"   Scan target position: Y={startY + targetYPositionOffset*1e-3:.3f} µm, Z={startZ + targetZPositionOffset*1e-3:.3f} µm")

# linear scan plotter
def plotLinear2DScan(filepath, resolution=200, cmap='inferno', targetYPositionOffset=0, targetZPositionOffset=0):
    # Load data
    df = pd.read_csv(filepath)
    y = df['y_mm'].values
    z = df['z_mm'].values
    r = df['r'].values
    startY = df['start_y_mm'].values[0]
    startZ = df['start_z_mm'].values[0]

    # Mask zeros and convert to microns
    mask = (y != 0)
    y_um = (y[mask] - y[mask].min()) * 1000
    z_um = (z[mask] - z[mask].min()) * 1000
    r = r[mask]

    # Starting position in micron coordinates
    startY_um = (startY - y[mask].min()) * 1000
    startZ_um = (startZ - z[mask].min()) * 1000

    # Interpolate onto regular grid
    y_grid, z_grid = np.meshgrid(np.linspace(y_um.min(), y_um.max(), resolution),
                                  np.linspace(z_um.min(), z_um.max(), resolution))
    R_grid = griddata((y_um, z_um), r, (y_grid, z_grid), method='linear')

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(R_grid,
                   extent=[y_um.min(), y_um.max(), z_um.min(), z_um.max()],
                   origin='lower',
                   aspect='equal',
                   cmap=cmap)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Photovoltage (V)')

    # Mark starting position
    ax.plot(startY_um, startZ_um, 'w+', markersize=15, markeredgewidth=3)
    ax.plot(startY_um, startZ_um, 'r*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Start')
    ax.legend(fontsize=8)

    if targetZPositionOffset != 0 or targetYPositionOffset != 0:
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'w+', markersize=15, markeredgewidth=3)
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'b*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Target')
        ax.legend(fontsize=8)

    ax.set_xlabel('Y Position (µm)')
    ax.set_ylabel('Z Position (µm)')
    ax.set_title(f'2D Scan: Photovoltage Map\n{filepath}')
    plt.tight_layout()
    plt.show()

    print(f"   Total points loaded: {len(r)}")
    print(f"   Y range: {y_um.min():.2f} to {y_um.max():.2f} µm")
    print(f"   Z range: {z_um.min():.2f} to {z_um.max():.2f} µm")
    print(f"   Scan center position: Y={startY:.3f} mm, Z={startZ:.3f} mm")
    if targetZPositionOffset != 0 or targetYPositionOffset != 0:
        print(f"   Scan target position: Y={startY + targetYPositionOffset*1e-3:.3f} mm, Z={startZ + targetZPositionOffset*1e-3:.3f} mm")

def plotLinear2DScanPcolormesh(filepath, cmap='viridis', targetYPositionOffset=0, targetZPositionOffset=0):
    # Load data
    df = pd.read_csv(filepath)
    y = df['y_mm'].values
    z = df['z_mm'].values
    r = df['r'].values
    n_y = int(df['n_y'].values[0])
    n_z = int(df['n_z'].values[0])
    startY = df['start_y_mm'].values[0]
    startZ = df['start_z_mm'].values[0]

    print(f"   Grid: {n_y} x {n_z} = {n_y*n_z}, data points: {len(r)}")

    Y = y.reshape(n_z, n_y).copy()
    Z = z.reshape(n_z, n_y).copy()
    R = r.reshape(n_z, n_y).copy()

    # No snake correction needed — linear scan always goes same direction

    # Convert to microns
    Y_um = (Y - Y.min()) * 1000
    Z_um = (Z - Z.min()) * 1000
    startY_um = (startY - y.min()) * 1000
    startZ_um = (startZ - z.min()) * 1000

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.pcolormesh(Y_um, Z_um, R, shading='auto', cmap=cmap)
    ax.set_aspect('equal')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Photovoltage (V)')

    # Mark starting position
    ax.plot(startY_um, startZ_um, 'w+', markersize=15, markeredgewidth=3)
    ax.plot(startY_um, startZ_um, 'r*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Start')
    ax.legend(fontsize=8)

    if targetZPositionOffset != 0 or targetYPositionOffset != 0:
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'w+', markersize=15, markeredgewidth=3)
        ax.plot(startY_um + targetYPositionOffset, startZ_um + targetZPositionOffset, 'b*', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Target')
        ax.legend(fontsize=8)

    ax.set_xlabel('Y Position (µm)')
    ax.set_ylabel('Z Position (µm)')
    ax.set_title(f'2D Scan: Photovoltage Map\n{filepath}')
    plt.tight_layout()
    plt.show()

    print(f"   Y range: {Y_um.min():.2f} to {Y_um.max():.2f} µm")
    print(f"   Z range: {Z_um.min():.2f} to {Z_um.max():.2f} µm")
    print(f"   Scan center position: Y={startY:.3f} mm, Z={startZ:.3f} mm")
    if targetZPositionOffset != 0 or targetYPositionOffset != 0:
        print(f"   Scan target position: Y={startY + targetYPositionOffset*1e-3:.3f} mm, Z={startZ + targetZPositionOffset*1e-3:.3f} mm")